In [2]:
from sklearn.datasets import fetch_openml

bunch = fetch_openml(data_id=45580, as_frame=True)
orange = bunch.frame
orange.shape

(11896, 180)

In [4]:
orange.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,FACTOR11,FACTOR12,FACTOR13,FACTOR14,FACTOR15,FACTOR16,FACTOR17,FACTOR18,y,t
0,-3.064007,-1.272259,-4.075844,1.492514,2.328039,-0.330240,-0.770389,0.035541,2.439325,-0.847266,...,V2,V2,V5,V20,V9,V11,V1,V4,0,0
1,-4.574066,-3.541815,1.107371,0.447314,-0.471140,-0.567309,0.195963,0.383654,1.523472,-0.184596,...,V2,V3,V6,V20,V9,V15,V4,V4,0,0
2,-1.751471,-2.039692,-3.788823,0.624226,0.564614,-0.713385,1.003502,1.582819,-1.461687,0.844975,...,V5,V2,V11,V20,V9,V1,V4,V4,0,1
3,-2.030089,-0.235720,-5.711960,1.071841,2.441541,0.758484,-0.534295,1.614289,-1.022909,-1.383025,...,V2,V2,V4,V20,V11,V1,V4,V4,0,1
4,-2.377857,-2.478670,-1.051946,2.812768,2.019267,0.095542,-0.033324,-1.332319,4.316526,-0.537967,...,V2,V2,V7,V20,V2,V1,V2,V2,0,1


In [3]:
import pandas as pd

HILLSTROM_URL = "http://www.minethatdata.com/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
hill = pd.read_csv(HILLSTROM_URL)
hill.shape

(64000, 12)

In [5]:
hill.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [14]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

def add_risk(df, higher_is_better, drop_cols=(), control="대조군", n_bins=3, seed=42):
    df = df.copy()
    df["bad"] = (1 - df["outcome"]) if higher_is_better else df["outcome"]

    feats = [c for c in df.columns if c not in ("group", "outcome", "bad", *drop_cols)]
    X = df[feats].copy()
    for c in X.select_dtypes(include=["object", "category"]).columns:
        X[c] = X[c].astype("category")

    is_ctrl = df["group"] == control
    model = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=seed, verbosity=-1)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    risk = pd.Series(index=df.index, dtype=float)
    risk[is_ctrl] = cross_val_predict(
        model, X[is_ctrl], df.loc[is_ctrl, "bad"], cv=skf, method="predict_proba"
    )[:, 1]
    model.fit(X[is_ctrl], df.loc[is_ctrl, "bad"])
    risk[~is_ctrl] = model.predict_proba(X[~is_ctrl])[:, 1]

    df["risk"] = risk
    df["risk_bin"] = pd.qcut(df["risk"].rank(method="first"), n_bins, labels=["저", "중", "고"])
    return df

In [15]:
hill_r = add_risk(hill_std, True, drop_cols=("conversion", "spend"))
orange_r = add_risk(orange_std, False)

In [17]:
for name, d in [("hillstrom", hill_r), ("orange", orange_r)]:
    print(name)
    display(d.groupby(["risk_bin", "group"], observed=True)["bad"].mean().unstack().round(4))

hillstrom


group,Mens E-Mail,Womens E-Mail,대조군
risk_bin,,,
저,0.7593,0.8029,0.8516
중,0.8161,0.8494,0.8878
고,0.8777,0.8941,0.9402


orange


group,대조군,연락
risk_bin,,
저,0.0069,0.0086
중,0.0298,0.0217
고,0.0949,0.0659


In [18]:
def rr_table(df, control="대조군"):
    rows = []
    for b, sub in df.groupby("risk_bin", observed=True):
        stat = sub.groupby("group")["bad"].agg(["size", "sum"])
        n0, e0 = stat.loc[control, "size"], stat.loc[control, "sum"]

        for g in stat.index.drop(control):
            n1, e1 = stat.loc[g, "size"], stat.loc[g, "sum"]
            if e0 == 0 or e1 == 0:
                e0, e1, n0, n1 = e0 + 0.5, e1 + 0.5, n0 + 1, n1 + 1

            rr = (e1 / n1) / (e0 / n0)
            se = np.sqrt(1/e1 - 1/n1 + 1/e0 - 1/n0)
            rows.append({
                "risk_bin": b, "campaign": g,
                "대조군 인원": n0, "실험군 인원": n1,
                "대조군 나쁜결과(%)": e0 / n0 * 100,
                "실험군 나쁜결과(%)": e1 / n1 * 100,
                "rr": rr,
                "rr_lo": np.exp(np.log(rr) - 1.96 * se),
                "rr_hi": np.exp(np.log(rr) + 1.96 * se),
            })
    return pd.DataFrame(rows)

In [19]:
effects = pd.concat([
    rr_table(hill_r).assign(source="hillstrom"),
    rr_table(orange_r).assign(source="orange"),
], ignore_index=True)

effects["campaign"] = effects["campaign"].replace({
    "Mens E-Mail": "이메일A", "Womens E-Mail": "이메일B", "연락": "콜센터 연락",
})
effects["변화율(%)"] = (effects["rr"] - 1) * 100
effects.round(3)

,risk_bin,campaign,대조군 인원,실험군 인원,대조군 나쁜결과(%),실험군 나쁜결과(%),rr,rr_lo,rr_hi,source,변화율(%)
0,저,이메일A,7057,7158,85.164,75.929,0.892,0.877,0.906,hillstrom,-10.843
1,저,이메일B,7057,7118,85.164,80.289,0.943,0.929,0.957,hillstrom,-5.723
2,중,이메일A,6924,7157,88.778,81.612,0.919,0.907,0.932,hillstrom,-8.072
3,중,이메일B,6924,7252,88.778,84.942,0.957,0.945,0.969,hillstrom,-4.321
4,고,이메일A,7325,6992,94.020,87.772,0.934,0.924,0.943,hillstrom,-6.646
5,고,이메일B,7325,7017,94.020,89.411,0.951,0.942,0.960,hillstrom,-4.902
6,저,콜센터 연락,1300,2665,0.692,0.863,1.247,0.578,2.686,orange,24.661
7,중,콜센터 연락,838,3127,2.983,2.175,0.729,0.464,1.146,orange,-27.107
8,고,콜센터 연락,748,3218,9.492,6.588,0.694,0.537,0.897,orange,-30.595


In [20]:
kk = pd.read_csv("../../data/processed/kkbox_scored.csv")
kk["risk_bin"] = pd.qcut(kk["churn_prob"].rank(method="first"), 3, labels=["저", "중", "고"])

In [21]:
def apply_campaign(kk, effects, campaign):
    eff = effects.loc[effects["campaign"] == campaign, ["risk_bin", "rr", "rr_lo", "rr_hi"]].copy()
    eff["risk_bin"] = eff["risk_bin"].astype(str)

    out = kk.copy()
    out["risk_bin"] = out["risk_bin"].astype(str)
    out = out.merge(eff, on="risk_bin", how="left")

    out["prob_after"]    = (out["churn_prob"] * out["rr"]).clip(upper=1)
    out["prob_after_lo"] = (out["churn_prob"] * out["rr_lo"]).clip(upper=1)
    out["prob_after_hi"] = (out["churn_prob"] * out["rr_hi"]).clip(upper=1)
    return out

In [22]:
test = apply_campaign(kk, effects, "콜센터 연락")
test.groupby("risk_bin")[["churn_prob", "prob_after", "prob_after_lo", "prob_after_hi"]].mean().round(4)

,churn_prob,prob_after,prob_after_lo,prob_after_hi
risk_bin,,,,
고,0.2115,0.1468,0.1135,0.1897
저,0.0189,0.0235,0.0109,0.0507
중,0.0386,0.0282,0.0179,0.0443


In [23]:
def segment_effect(kk, effects, mask, labels=("부류A", "부류B")):
    rows = []
    for camp in effects["campaign"].unique():
        out = apply_campaign(kk, effects, camp)
        out["segment"] = np.where(mask.values, labels[0], labels[1])

        g = out.groupby("segment").agg(
            인원=("churn_prob", "size"),
            현재=("churn_prob", "sum"),
            후=("prob_after", "sum"),
            낙관=("prob_after_lo", "sum"),
            비관=("prob_after_hi", "sum"),
        ).reset_index()
        g["campaign"] = camp
        rows.append(g)

    res = pd.concat(rows, ignore_index=True)
    res["감소 인원"] = res["현재"] - res["후"]
    res["감소 범위"] = (res["현재"] - res["비관"]).round(0).astype(int).astype(str) + " ~ " + \
                      (res["현재"] - res["낙관"]).round(0).astype(int).astype(str)
    res["변화율(%)"] = (res["후"] / res["현재"] - 1) * 100
    return res.round(1)

In [25]:
mask = kk["last_auto_renew"] == 0
res = segment_effect(kk, effects, mask, labels=("자동갱신 끔", "자동갱신 켬"))
res

,segment,인원,현재,후,낙관,비관,campaign,감소 인원,감소 범위,변화율(%)
0,자동갱신 끔,11810,4695.8,4383.7,4338.0,4429.9,이메일A,312.1,266 ~ 358,-6.6
1,자동갱신 켬,88190,4269.8,3941.2,3893.0,3990.1,이메일A,328.5,280 ~ 377,-7.7
2,자동갱신 끔,11810,4695.8,4465.6,4421.6,4510.1,이메일B,230.2,186 ~ 274,-4.9
3,자동갱신 켬,88190,4269.8,4062.8,4016.1,4109.9,이메일B,207.0,160 ~ 254,-4.8
4,자동갱신 끔,11810,4695.8,3259.2,2521.3,4213.0,콜센터 연락,1436.7,483 ~ 2175,-30.6
5,자동갱신 켬,88190,4269.8,3355.9,2224.5,5275.8,콜센터 연락,913.9,-1006 ~ 2045,-21.4


In [26]:
res.pivot(index="campaign", columns="segment", values="감소 인원")

segment,자동갱신 끔,자동갱신 켬
campaign,,
이메일A,312.1,328.5
이메일B,230.2,207.0
콜센터 연락,1436.7,913.9


In [27]:
pd.crosstab(np.where(mask, "자동갱신 끔", "자동갱신 켬"), kk["risk_bin"], normalize="index").round(3)

risk_bin,저,중,고
row_0,,,
자동갱신 끔,0.000,0.000,1.000
자동갱신 켬,0.378,0.378,0.244


In [28]:
mask = kk["bd_clean"] < 25                  # 25세 미만 vs 이상
mask = kk["tenure_date"] < 365              # 가입 1년 미만 vs 이상
mask = kk["churn_prob"] >= 0.280            # 위험 고객 vs 나머지

In [29]:
SB = "../../data/raw/starbucks/"
portfolio  = pd.read_json(SB + "portfolio.json",  orient="records", lines=True)
profile    = pd.read_json(SB + "profile.json",    orient="records", lines=True)
transcript = pd.read_json(SB + "transcript.json", orient="records", lines=True)

for ch in ["email", "mobile", "social", "web"]:
    portfolio[ch] = portfolio["channels"].apply(lambda x: int(ch in x))

portfolio["benefit"] = np.where(
    portfolio["difficulty"] > 0, portfolio["reward"] / portfolio["difficulty"], 0
)

In [30]:
profile["age"] = profile["age"].replace(118, np.nan)
profile["became_member_on"] = pd.to_datetime(profile["became_member_on"].astype(str), format="%Y%m%d")
profile["tenure_days"] = (profile["became_member_on"].max() - profile["became_member_on"]).dt.days

In [31]:
val = pd.json_normalize(transcript["value"])
transcript["offer_id"] = val["offer id"].fillna(val["offer_id"])
transcript["amount"] = val["amount"]
transcript["day"] = transcript["time"] / 24

In [32]:
transcript["event"].value_counts()
transcript.loc[transcript["event"] == "offer received", "time"].unique()

array([  0, 168, 336, 408, 504, 576])

In [34]:
W = 72   # 관찰 기간 (시간) = 3일
waves = sorted(transcript.loc[transcript["event"] == "offer received", "time"].unique())

rec = transcript.loc[transcript["event"] == "offer received", ["person", "time", "offer_id"]]
tx  = transcript.loc[transcript["event"] == "transaction", ["person", "time", "amount"]]

rows = []
for w in waves:
    base = profile[["id"]].rename(columns={"id": "person"})
    base["wave"] = w

    r = rec.loc[rec["time"] == w, ["person", "offer_id"]].drop_duplicates("person")
    base = base.merge(r, on="person", how="left")

    t = (tx[(tx["time"] >= w) & (tx["time"] < w + W)]
         .groupby("person").agg(n_tx=("amount", "size"), spend=("amount", "sum")))
    base = base.merge(t, on="person", how="left").fillna({"n_tx": 0, "spend": 0})
    rows.append(base)

panel = pd.concat(rows, ignore_index=True)

In [35]:
offer_cols = ["id", "offer_type", "email", "mobile", "social", "web", "benefit", "duration"]
panel = panel.merge(portfolio[offer_cols], left_on="offer_id", right_on="id", how="left").drop(columns="id")

panel["offer_type"] = panel["offer_type"].fillna("none")
panel[["email", "mobile", "social", "web", "benefit", "duration"]] = \
    panel[["email", "mobile", "social", "web", "benefit", "duration"]].fillna(0)

panel = panel.merge(profile[["id", "age", "gender", "income", "tenure_days"]],
                    left_on="person", right_on="id", how="left").drop(columns="id")

In [36]:
panel.shape
panel.groupby("offer_type")["n_tx"].agg(["size", "mean"]).round(3)

,size,mean
offer_type,,
bogo,30499,1.046
discount,30543,0.962
informational,15235,0.900
none,25723,0.594


In [37]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

panel["gender"] = panel["gender"].astype("category")
panel["offer_type"] = panel["offer_type"].astype("category")

feat_cust  = ["age", "gender", "income", "tenure_days"]
feat_offer = ["offer_type", "email", "mobile", "social", "web", "benefit", "duration"]
FEATS = feat_cust + feat_offer

beh_model = LGBMRegressor(
    objective="poisson", n_estimators=300, learning_rate=0.05,
    num_leaves=31, min_child_samples=50, random_state=42, verbosity=-1,
)

In [38]:
gkf = GroupKFold(n_splits=5)
pred = cross_val_predict(beh_model, panel[FEATS], panel["n_tx"], cv=gkf, groups=panel["person"])

check = panel.assign(pred=pred).groupby("offer_type", observed=True)[["n_tx", "pred"]].mean().round(3)
check

,n_tx,pred
offer_type,,
bogo,1.046,1.045
discount,0.962,0.957
informational,0.900,0.902
none,0.594,0.599


In [39]:
beh_model.fit(panel[FEATS], panel["n_tx"])

,learning_rate,0.05
,n_estimators,300
,objective,'poisson'
,min_child_samples,50
,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None


In [40]:
TYPE_MAP = {"할인": "discount", "무료 기간 연장": "bogo", "콘텐츠 추천": "informational", "단순 안내": "informational"}

def campaign_to_offer(channel, kind, benefit_pct, days):
    offer = {"email": 0, "mobile": 0, "social": 0, "web": 0}
    offer["email" if channel == "이메일" else "mobile"] = 1
    offer["offer_type"] = TYPE_MAP[kind]
    offer["benefit"] = 0 if TYPE_MAP[kind] == "informational" else benefit_pct / 100
    offer["duration"] = days
    return offer

In [41]:
kk_cust = pd.DataFrame({
    "age": kk["bd_clean"],
    "gender": kk["gender_status"].map({"male": "M", "female": "F"}),
    "income": np.nan,
    "tenure_days": kk["tenure_date"],
})

In [49]:
def behavior_lift(kk_cust, offer):
    x_on = kk_cust.copy()
    for k, v in offer.items():
        x_on[k] = v

    x_off = kk_cust.copy()
    x_off[["email", "mobile", "social", "web", "benefit", "duration"]] = 0
    x_off["offer_type"] = "none"

    for x in (x_on, x_off):
        x["gender"] = pd.Categorical(x["gender"], categories=panel["gender"].cat.categories)
        x["offer_type"] = pd.Categorical(x["offer_type"], categories=panel["offer_type"].cat.categories)

    on = beh_model.predict(x_on[FEATS])
    off = beh_model.predict(x_off[FEATS])
    return on / off

In [50]:
offer = campaign_to_offer("앱 푸시", "할인", 30, 7)
kk["lift"] = behavior_lift(kk_cust, offer)
kk["lift"].describe()

count    100000.000000
mean          1.223344
std           0.160909
min           0.909119
25%           1.083128
50%           1.234151
75%           1.335410
max           1.626354
Name: lift, dtype: float64

In [59]:
exclude = ["msno", "churn_prob", "is_churn", "risk_bin", "lift", "prob_before", "prob_after"]
feat_cols = [c for c in kk.columns if c not in exclude]

cats = [c for c in ["registered_via", "city_clean", "gender_status", "reg_season"] if c in feat_cols]
nums = [c for c in feat_cols if c not in cats]

cats, len(nums)

(['registered_via', 'city_clean', 'gender_status'], 30)

In [60]:
for c in cats:
    kk[c] = kk[c].astype("category")

churn_model = LGBMClassifier(
    n_estimators=100, learning_rate=0.0302, num_leaves=31,
    min_child_samples=76, random_state=42, verbosity=-1,
).fit(kk[cats + nums], kk["is_churn"])

In [56]:
def compare_groups(kk, mask, labels=("부류A", "부류B")):
    seg = np.where(mask, labels[0], labels[1])
    g = kk.assign(segment=seg).groupby("segment").agg(
        인원=("prob_before", "size"),
        평균_활동변화=("lift", "mean"),
        현재_예상이탈=("prob_before", "sum"),
        캠페인후_예상이탈=("prob_after", "sum"),
    )
    g["감소 인원"] = g["현재_예상이탈"] - g["캠페인후_예상이탈"]
    g["변화율(%)"] = (g["캠페인후_예상이탈"] / g["현재_예상이탈"] - 1) * 100
    return g.round(2)

In [62]:
def apply_behavior(x, lift, discount=0.0, extend_days=0):
    x = x.copy()
    has = lambda cols: [c for c in cols if c in x.columns]

    # 청취 활동: 행동 변화율만큼 늘리기
    act = has(["total_secs", "total_num_100", "total_num_unq"])
    x[act] = x[act].mul(lift, axis=0)
    if "activity_days" in x:
        x["activity_days"] = (x["activity_days"] * lift).clip(upper=90).round()
    if "days_since_last_log" in x:
        x["days_since_last_log"] = (x["days_since_last_log"] / lift).round()

    # 함께 계산된 컬럼 다시 맞추기
    if "activity_days" in x:
        d = x["activity_days"].replace(0, np.nan)
        for new, base in [("avg_daily_secs", "total_secs"),
                          ("avg_daily_complete", "total_num_100"),
                          ("avg_daily_unq", "total_num_unq")]:
            if new in x and base in x:
                x[new] = x[base] / d

    # 할인: 결제 금액 낮추기
    pay = has(["avg_payment", "payment_per_plan_day"])
    x[pay] = x[pay] * (1 - discount)

    # 무료 기간 연장: 이용권 기간 늘리기
    if "last_plan_days" in x:
        x["last_plan_days"] += extend_days
    return x

In [63]:
X = kk[cats + nums]
X_after = apply_behavior(X, kk["lift"].values, discount=0.3)

kk["prob_before"] = churn_model.predict_proba(X)[:, 1]
kk["prob_after"]  = churn_model.predict_proba(X_after)[:, 1]

kk[["prob_before", "prob_after"]].mean()

prob_before    0.089595
prob_after     0.153299
dtype: float64

In [64]:
ones = np.ones(len(X))
tests = {
    "변경 없음":     X,
    "청취 활동만":   apply_behavior(X, kk["lift"].values, discount=0),
    "할인만":        apply_behavior(X, ones, discount=0.3),
}
for name, xa in tests.items():
    print(f"{name:10s} {churn_model.predict_proba(xa)[:, 1].mean():.4f}")

변경 없음      0.0896
청취 활동만     0.0882
할인만        0.1548


In [65]:
X_after = apply_behavior(X, kk["lift"].values, discount=0)

kk["prob_before"] = churn_model.predict_proba(X)[:, 1]
kk["prob_after"]  = churn_model.predict_proba(X_after)[:, 1]
kk[["prob_before", "prob_after"]].mean()

prob_before    0.089595
prob_after     0.088177
dtype: float64

In [48]:
mask = kk["last_auto_renew"] == 0
compare_groups(kk, mask, labels=("자동갱신 끔", "자동갱신 켬"))

KeyError: "Column(s) ['prob_after', 'prob_before'] do not exist"

In [66]:
def simulate(kk, kk_cust, X, channel, kind, benefit_pct, days, mask, labels):
    offer = campaign_to_offer(channel, kind, benefit_pct, days)
    lift = behavior_lift(kk_cust, offer)
    X_after = apply_behavior(X, lift, discount=0)

    out = kk.assign(
        lift=lift,
        prob_before=churn_model.predict_proba(X)[:, 1],
        prob_after=churn_model.predict_proba(X_after)[:, 1],
    )
    return compare_groups(out, mask, labels)

In [67]:
mask = kk["last_auto_renew"] == 0
labels = ("자동갱신 끔", "자동갱신 켬")

for c in [("앱 푸시", "단순 안내", 0, 3), ("앱 푸시", "할인", 30, 7), ("이메일", "할인", 30, 7)]:
    print(c)
    display(simulate(kk, kk_cust, X, *c, mask, labels))

('앱 푸시', '단순 안내', 0, 3)


,인원,평균_활동변화,현재_예상이탈,캠페인후_예상이탈,감소 인원,변화율(%)
segment,,,,,,
자동갱신 끔,11810,1.19,4697.13,4595.36,101.77,-2.17
자동갱신 켬,88190,1.19,4262.37,4232.97,29.40,-0.69


('앱 푸시', '할인', 30, 7)


,인원,평균_활동변화,현재_예상이탈,캠페인후_예상이탈,감소 인원,변화율(%)
segment,,,,,,
자동갱신 끔,11810,1.21,4697.13,4587.80,109.33,-2.33
자동갱신 켬,88190,1.22,4262.37,4229.92,32.45,-0.76


('이메일', '할인', 30, 7)


,인원,평균_활동변화,현재_예상이탈,캠페인후_예상이탈,감소 인원,변화율(%)
segment,,,,,,
자동갱신 끔,11810,1.25,4697.13,4570.68,126.45,-2.69
자동갱신 켬,88190,1.23,4262.37,4229.13,33.24,-0.78
